# What's in the lakehouse

A look at every Delta table that exists right now: what's there, how big it is, when it was last written, and what's inside bronze.

Tables are **discovered**, not listed by hand — any folder under the lakehouse root with a `_delta_log` is a Delta table — so this notebook keeps working as Phase 2 adds silver and Phase 5 adds gold.

Read-only: nothing here writes to a table.

In [ ]:
import json
import os

import pandas as pd
from pyspark.sql import functions as F

from ingestion import paths
from ingestion.spark import get_spark

spark = get_spark("lakehouse-tables")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 80)

## 1. Catalog: every table, one row each

`DESCRIBE DETAIL` reads Delta's metadata, not the data — so it's instant even on a big table. `numFiles` is worth watching: thousands of tiny files is the "small files problem" that slows Spark down.

In [ ]:
def find_delta_tables(root):
    for folder, subfolders, _ in os.walk(root):
        if "_delta_log" in subfolders:
            subfolders.clear()  # don't descend into a table's own folders
            yield folder

catalog = []
for path in sorted(find_delta_tables(paths.LAKEHOUSE_ROOT)):
    detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").first()
    catalog.append({
        "table": os.path.relpath(path, paths.LAKEHOUSE_ROOT),
        "rows": spark.read.format("delta").load(path).count(),
        "files": detail.numFiles,
        "size_kb": round(detail.sizeInBytes / 1024, 1),
        "last_write_utc": detail.lastModified,
        "path": path,
    })
catalog = pd.DataFrame(catalog)
catalog.drop(columns="path")

## 2. Schemas

The declared schema of each table. `nullable = false` is a constraint Delta enforces on every write.

In [ ]:
for row in catalog.itertuples():
    print(f"── {row.table}")
    spark.read.format("delta").load(row.path).printSchema()

## 3. Commit history

Every write is a numbered commit. Useful for "what ran, when, and what did it do?" — and the basis of time travel (`versionAsOf`).

In [ ]:
for row in catalog.itertuples():
    print(f"── {row.table}")
    spark.sql(f"DESCRIBE HISTORY delta.`{row.path}`").select(
        "version", "timestamp", "operation", "operationMetrics.numOutputRows"
    ).show(10, truncate=False)

## 4. Bronze: what was loaded, by whom, for which dates

One row per vendor response. Grouping by `run_id` shows each load; a date fetched twice shows up as two runs — bronze keeps both deliveries on purpose.

In [ ]:
bronze = {source: spark.read.format("delta").load(path) for source, path in paths.BRONZE_BARS.items()
          if os.path.isdir(os.path.join(path, "_delta_log"))}

for source, df in bronze.items():
    print(f"── {source}")
    df.groupBy("run_id", "request_start", "request_end").agg(
        F.count("*").alias("responses"),
        F.countDistinct("security_id").alias("securities"),
        F.sum("record_count").alias("bars"),
        F.min("ingested_at").alias("ingested_at"),
    ).orderBy("ingested_at").show(truncate=False)

Responses that came back **empty** (`record_count = 0`). On a trading day this is a missing security; silver's completeness check (step 3) will turn it into a failure.

In [ ]:
for source, df in bronze.items():
    empty = df.where("record_count = 0").select("security_id", "vendor_symbol", "request_start", "run_id")
    print(f"{source}: {empty.count()} empty responses")
    empty.show(10, truncate=False)

## 5. Inside one payload

The raw text each vendor sent for Berkshire Class B, plus the request that produced it. Note the two spellings (`BRK.B` / `BRK-B`) — both rows carry the same `security_id`.

In [ ]:
BRK_B = "BBG000DWG505"
for source, df in bronze.items():
    r = df.where(F.col("security_id") == BRK_B).orderBy(F.desc("ingested_at")).first()
    if r is None:
        continue
    print(f"── {source}  (as '{r.vendor_symbol}', run {r.run_id})")
    print("request:", r.request_params)
    print("payload:", r.payload[:400], "\n")

## 6. The two vendors side by side

A **quick look only** — the payloads are parsed here just to display them. The real parsing (types, `trade_date` in New York time, decimals) belongs in silver, step 3, with tests; if this cell ever grows into pipeline logic, it's in the wrong place.

Prices should agree to the cent. Volumes usually don't, quite: the vendors count shares slightly differently, which is exactly the divergence Phase 4 will measure.

In [ ]:
def quick_parse(source, payload, symbol):
    body = json.loads(payload)
    if source == "alpaca":
        return [{"t": b["t"], "close": b["c"], "volume": b["v"]} for b in (body["bars"] or {}).get(symbol, [])]
    cols = body["columns"]
    return [{"t": t, "close": row[cols.index("Close")], "volume": row[cols.index("Volume")]}
            for t, row in zip(body["index"], body["data"])]

frames = []
for source, df in bronze.items():
    latest = df.groupBy("security_id").agg(F.max("ingested_at").alias("ingested_at"))
    for r in df.join(latest, ["security_id", "ingested_at"]).collect():
        for bar in quick_parse(source, r.payload, r.vendor_symbol):
            frames.append({"security_id": r.security_id, "source": source, **bar})

bars = pd.DataFrame(frames)
side = bars.pivot_table(index=["security_id", "t"], columns="source", values=["close", "volume"]).dropna()
side["close_diff"] = (side[("close", "alpaca")] - side[("close", "yahoo")]).round(4)
side["volume_diff_pct"] = ((side[("volume", "alpaca")] / side[("volume", "yahoo")] - 1) * 100).round(2)
print(f"{len(side)} security-days with both vendors")
print(f"closes differing by more than half a cent: {(side['close_diff'].abs() > 0.005).sum()}")
side.sort_values("volume_diff_pct", key=abs, ascending=False).head(10)

In [ ]:
spark.stop()